# LLM Drift Monitor: reproducible Colab walkthrough

This notebook runs the **current repository code** in a fresh Colab CPU runtime.
It clones GitHub, installs `requirements.txt`, and calls the real
`LLMDriftMonitor` pipeline. It does not reimplement the statistics or upload
modules by hand.

Three run modes are supported:

- **Live Gemini (primary):** `GEMINI_API_KEY` from Colab Secrets or the
  environment, `gemini-3.1-flash-lite` through the native `generateContent`
  adapter. This is not the OpenAI-compatible endpoint.
- **Optional live providers:** Groq or xAI/Grok through the genuinely supported
  `openai-compatible` adapter.
- **Controlled simulation:** if no credential is available, the repository's
  `FakeLLMAdapter` is used and every result is labelled as simulation.

Never paste an API key into a cell. Run cells top to bottom.

## 1. Clone the source and install its dependencies

The GitHub repository is authoritative. The notebook reuses its modules and
the committed 40-prompt panel. The clone destination and source can be
overridden with environment variables for local validation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get(
    "DRIFT_MONITOR_REPO_URL",
    "https://github.com/chandresh080/llm-drift-monitor.git",
).strip()
REPO_DIR = Path(os.environ.get(
    "DRIFT_MONITOR_REPO_DIR", "/content/llm-drift-monitor"
)).expanduser()

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a git checkout; choose a different path."
    )
if not (REPO_DIR / ".git").exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )
else:
    print(f"Reusing existing checkout: {REPO_DIR}")

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

required = [
    "requirements.txt",
    "monitor.py",
    "storage.py",
    "embeddings.py",
    "drift_tests.py",
    "behavior_features.py",
    "sequential_testing.py",
    "colab/config.yaml",
    "config/canary_prompts.json",
]
missing = [name for name in required if not (REPO_DIR / name).exists()]
if missing:
    raise RuntimeError(f"Checkout is missing required files: {missing}")

print("Repository:", REPO_DIR)
print("Commit:", commit)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--disable-pip-version-check", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Installed dependencies from", REPO_DIR / "requirements.txt")

## 2. Runtime configuration and provider selection

`colab/config.yaml` is intentionally small: 10 probes per cycle and a 30-run
reference window. The hosted deployment uses 40 probes, a 120-run reference
window and PostgreSQL; those are not copied here.

`DATABASE_URL` is cleared so a fresh Colab run cannot write to the hosted
database. The embedding backend is forced to ONNX to avoid loading a heavy
torch model on a free CPU runtime.

In [ ]:
import yaml

os.chdir(REPO_DIR)  # monitor.py loads config/canary_prompts.json from cwd
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ.pop("DATABASE_URL", None)
os.environ["DRIFT_MONITOR_EMBEDDING_BACKEND"] = "onnx"
os.environ["DRIFT_MONITOR_TRACKING"] = "0"

SOURCE_CONFIG = REPO_DIR / "colab" / "config.yaml"
RUNTIME_CONFIG = Path(os.environ.get(
    "DRIFT_MONITOR_CONFIG_PATH", "/content/drift_monitor_runtime.yaml"
))
DB_PATH = Path(os.environ.get("DRIFT_MONITOR_DB_PATH", "/content/drift_monitor.db"))
RUNTIME_CONFIG.parent.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

if os.environ.get("DRIFT_MONITOR_RESET_DB", "1") == "1" and DB_PATH.exists():
    DB_PATH.unlink()

config = yaml.safe_load(SOURCE_CONFIG.read_text())
config["data"]["db_path"] = str(DB_PATH)
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))

print("Runtime config:", RUNTIME_CONFIG)
print("Local DB:", DB_PATH)
print("Embedding backend:", os.environ["DRIFT_MONITOR_EMBEDDING_BACKEND"])
print("Tracking disabled:", os.environ["DRIFT_MONITOR_TRACKING"])

In [ ]:
def read_secret(name: str) -> str:
    """Read a Colab Secret, falling back to the process environment."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    return os.environ.get(name, "").strip()


KEY_NAMES = ("GEMINI_API_KEY", "GROQ_API_KEY", "XAI_API_KEY", "OPENAI_API_KEY")
MODEL_ENV_NAMES = (
    "GEMINI_MODEL", "GEMINI_BASE_URL",
    "GROQ_MODEL", "GROQ_BASE_URL",
    "XAI_MODEL", "XAI_BASE_URL",
    "OPENAI_MODEL", "OPENAI_BASE_URL",
)
secrets = {name: read_secret(name) for name in KEY_NAMES}
env_defaults = {name: os.environ.get(name) for name in MODEL_ENV_NAMES}

# Make the explicit selector authoritative, including when several keys exist.
for name in KEY_NAMES + MODEL_ENV_NAMES:
    os.environ.pop(name, None)

choice = os.environ.get("DRIFT_MONITOR_PROVIDER", "auto").strip().lower()
valid_choices = {"auto", "gemini", "groq", "xai", "fake"}
if choice not in valid_choices:
    raise ValueError(f"DRIFT_MONITOR_PROVIDER must be one of {sorted(valid_choices)}")

if choice == "auto":
    if secrets["GEMINI_API_KEY"]:
        choice = "gemini"
    elif secrets["GROQ_API_KEY"]:
        choice = "groq"
    elif secrets["XAI_API_KEY"] or secrets["OPENAI_API_KEY"]:
        choice = "xai"
    else:
        choice = "fake"

runtime_config = yaml.safe_load(RUNTIME_CONFIG.read_text())
target = runtime_config["target"]

if choice == "gemini":
    if not secrets["GEMINI_API_KEY"]:
        raise RuntimeError(
            "GEMINI_API_KEY is required for the native Gemini provider. "
            "Add it to Colab Secrets or set the environment variable."
        )
    os.environ["GEMINI_API_KEY"] = secrets["GEMINI_API_KEY"]
    target["type"] = "gemini"
    target["model_id"] = target.get("model_id") or "gemini-3.1-flash-lite"
    run_mode = "live-gemini-native"
elif choice == "groq":
    if not secrets["GROQ_API_KEY"]:
        raise RuntimeError("GROQ_API_KEY is required for the Groq provider.")
    os.environ["GROQ_API_KEY"] = secrets["GROQ_API_KEY"]
    groq_model = env_defaults.get("GROQ_MODEL") or "openai/gpt-oss-20b"
    os.environ["GROQ_MODEL"] = groq_model
    if env_defaults.get("GROQ_BASE_URL"):
        os.environ["GROQ_BASE_URL"] = env_defaults["GROQ_BASE_URL"]
    target["type"] = "openai-compatible"
    target["model_id"] = groq_model
    run_mode = "live-groq-openai-compatible"
elif choice == "xai":
    xai_key = secrets["XAI_API_KEY"] or secrets["OPENAI_API_KEY"]
    if not xai_key:
        raise RuntimeError("XAI_API_KEY or OPENAI_API_KEY is required for Grok.")
    os.environ["XAI_API_KEY"] = xai_key
    os.environ["OPENAI_API_KEY"] = xai_key
    xai_model = (env_defaults.get("OPENAI_MODEL")
                 or env_defaults.get("XAI_MODEL") or "grok-4.7")
    os.environ["OPENAI_MODEL"] = xai_model
    os.environ["XAI_MODEL"] = xai_model
    xai_base = (env_defaults.get("OPENAI_BASE_URL")
                or env_defaults.get("XAI_BASE_URL") or "https://api.x.ai/v1")
    os.environ["OPENAI_BASE_URL"] = xai_base
    os.environ["XAI_BASE_URL"] = xai_base
    target["type"] = "openai-compatible"
    target["model_id"] = xai_model
    run_mode = "live-xai-openai-compatible"
else:
    target["type"] = "fake"
    target["model_id"] = "fake-llm-v1"
    target["call_spacing_seconds"] = 0.0
    run_mode = "simulation-fake"

IS_SIMULATION = run_mode == "simulation-fake"
RUNTIME_CONFIG.write_text(yaml.safe_dump(runtime_config, sort_keys=False))

print("Run mode:", run_mode)
print("Adapter type:", target["type"])
print("Configured model:", target["model_id"])
print("Call spacing (s):", target.get("call_spacing_seconds", 0.0))
if IS_SIMULATION:
    print("SIMULATION: no provider credential was available. "
          "All downstream results are controlled fake-adapter output, "
          "not live provider measurements.")

## 3. Instantiate the monitor and check the runtime

The adapter is created from the runtime config. A failed preflight is recorded
rather than crashing: later cells will skip cycles and report that no detector
results were produced.

In [ ]:
from IPython.display import display
import pandas as pd

from monitor import LLMDriftMonitor

monitor = LLMDriftMonitor(config_path=str(RUNTIME_CONFIG))

BATCH_SIZE = int(monitor.config["monitoring"]["batch_size"])
REF_WINDOW = int(monitor.config["reference"]["window_size"])
ALPHA = float(monitor.config["alerts"]["significance_threshold"])
MIN_CONSECUTIVE = int(monitor.config["alerts"].get("min_consecutive_cycles", 1))

CYCLES = []
FAILED_CYCLES = []
PROVIDER_OK = monitor.adapter is not None

print("Run mode:", run_mode)
print("Adapter:", type(monitor.adapter).__name__ if monitor.adapter else None)
print("Model:", getattr(monitor.adapter, "model_id", None))
print("Prompt panel:", monitor.canary_prompts["version"],
      "-", len(monitor.canary_prompts["prompts"]), "prompts")
print("Batch size / reference window:", BATCH_SIZE, "/", REF_WINDOW)
print("Alpha / persistence:", ALPHA, "/", MIN_CONSECUTIVE)

storage_info = monitor.storage.info()
print("Storage backend:", storage_info["backend"])
print("Storage durable:", storage_info["durable"])
print("Storage location:", storage_info["location"])
print("Storage accepts writes:", storage_info["accepts_writes"])
if not storage_info["durable"]:
    print("Note: this is runtime-local SQLite; it may disappear on reset.")

if monitor.adapter is None:
    print("Provider unavailable:", monitor.adapter_error)
    PROVIDER_OK = False
else:
    try:
        ping_ok = bool(monitor.adapter.ping())
    except Exception as exc:
        ping_ok = False
        print("Provider preflight error:", type(exc).__name__, str(exc)[:200])
    PROVIDER_OK = ping_ok
    print("Provider preflight:", "ok" if ping_ok else "failed")


def safe_error(exc: BaseException, limit: int = 240) -> str:
    text = " ".join(str(exc).split())
    return text[:limit]


def cycle_counts(cycle: dict) -> dict:
    run_ids = set(cycle.get("run_ids") or [])
    rows = [r for r in monitor.storage.get_all_runs_ordered()
            if r.get("id") in run_ids]
    counts = {"total": len(rows), "success": 0, "empty": 0, "failed": 0}
    for row in rows:
        status = row.get("run_status") or "unknown"
        counts[status] = counts.get(status, 0) + 1
    return counts


def fmt(value, digits: int = 4) -> str:
    if value is None:
        return "n/a"
    try:
        if pd.isna(value):
            return "n/a"
    except (TypeError, ValueError):
        pass
    return f"{float(value):.{digits}f}"


def show_cycle(label: str, cycle: dict | None) -> None:
    if cycle is None:
        return
    counts = cycle_counts(cycle)
    outcome = ", ".join(f"{key}={counts.get(key, 0)}"
                        for key in ("success", "empty", "failed"))
    print(f"[{label}] cycle_id={cycle.get('cycle_id')} "
          f"tests_run={cycle.get('tests_run')} "
          f"reason={cycle.get('reason', '-')} | {outcome}")
    if cycle.get("tests_run"):
        result = cycle["results"]
        print("  reference/current:",
              result.get("reference_size"), "/", result.get("current_size"))
        print("  adjusted p-values: length=" + fmt(result.get("ks_adjusted_pvalue"))
              + ", availability=" + fmt(result.get("proportion_z_adjusted_pvalue"))
              + ", MMD=" + fmt(result.get("mmd_adjusted_pvalue")))


def run_cycle(label: str) -> dict | None:
    if not PROVIDER_OK:
        print(f"[{label}] skipped: provider unavailable")
        return None
    try:
        cycle = monitor.run_monitoring_cycle(num_probes=BATCH_SIZE)
    except Exception as exc:
        FAILED_CYCLES.append({"cycle": label, "error": type(exc).__name__,
                              "detail": safe_error(exc)})
        print(f"[{label}] provider error: {type(exc).__name__}: "
              f"{safe_error(exc)}")
        return None
    CYCLES.append((label, cycle))
    show_cycle(label, cycle)
    return cycle

## 4. Build the reference window first

The detectors compare two disjoint windows of **matched prompt IDs** from the
same panel version. A single cycle cannot do that: the first cycles create the
reference pool, and only a later cycle can supply a disjoint current window.
The `tests_run: false` messages below are the expected insufficient-sample
path, not a failure.

With this config: 3 cycles x 10 probes = 30 reference runs. The next cycle
supplies 10 current runs.

In [ ]:
REFERENCE_CYCLES = int(os.environ.get("DRIFT_MONITOR_REFERENCE_CYCLES", "3"))
for index in range(1, REFERENCE_CYCLES + 1):
    run_cycle(f"reference-{index}")

successes = monitor.storage.count_successful_runs()
print("Successful stored runs:", successes)
print("has_sufficient_reference:", monitor.has_sufficient_reference())
print("Evaluated cycles so far:",
      sum(1 for _, cycle in CYCLES if cycle.get("tests_run")))

## 5. Run evaluation cycles

Each evaluation cycle calls `run_monitoring_cycle`, which probes the adapter,
stores probe rows, runs the project's detectors, applies BH correction, and
generates alerts using the project's persistence rule. If the first evaluation
cycle still lacks matched data, a few extra cycles are attempted.

In [ ]:
EVALUATION_CYCLES = int(os.environ.get("DRIFT_MONITOR_EVAL_CYCLES", "2"))
for index in range(1, EVALUATION_CYCLES + 1):
    run_cycle(f"eval-{index}")

if PROVIDER_OK and not any(c.get("tests_run") for _, c in CYCLES):
    for index in range(1, 3):
        run_cycle(f"extra-{index}")

EVALUATED_CYCLES = [(label, cycle) for label, cycle in CYCLES
                    if cycle.get("tests_run")]
print("Evaluated cycles:", len(EVALUATED_CYCLES))
print("Failed cycles:", len(FAILED_CYCLES))

## 6. Detector results

The significance family is the project's prompt-aware tests: a paired mean
length contrast, a within-prompt validity permutation, and an
allocation-preserving prompt-cluster MMD. Their p-values receive
Benjamini-Hochberg correction inside each cycle.

`ks_statistic` in the stored payload is the paired t statistic, not the KS D
statistic. The pooled KS D is reported separately as
`ks_statistic_pooled` and is exploratory only.

In [ ]:
detector_rows = []
for label, cycle in EVALUATED_CYCLES:
    result = cycle["results"]
    detector_rows.append({
        "cycle": label,
        "cycle_id": cycle.get("cycle_id"),
        "ref_n": result.get("reference_size"),
        "cur_n": result.get("current_size"),
        "length_stat": fmt(result.get("ks_statistic")),
        "length_pooled_ks_D": fmt(result.get("ks_statistic_pooled")),
        "length_p_adj": fmt(result.get("ks_adjusted_pvalue")),
        "availability_stat": fmt(result.get("proportion_z_stat")),
        "availability_method": result.get("proportion_test_method"),
        "availability_p_adj": fmt(result.get("proportion_z_adjusted_pvalue")),
        "mmd_stat": fmt(result.get("mmd_statistic"), 6),
        "mmd_pooled_stat": fmt(result.get("mmd_statistic_pooled"), 6),
        "mmd_p_adj": fmt(result.get("mmd_adjusted_pvalue")),
        "BH flags": ", ".join(result.get("bh_significant_tests") or []) or "none",
        "distribution_shift": bool(result.get("distribution_shift_detected")),
        "quality_change": bool(result.get("quality_change_detected")),
        "quality_guardrail": bool(result.get("overall_quality_degradation")),
    })

if detector_rows:
    display(pd.DataFrame(detector_rows))
else:
    print("No evaluated cycles were produced; detector table is empty.")
if FAILED_CYCLES:
    display(pd.DataFrame(FAILED_CYCLES))

## 7. Effect sizes and separation of signals

Effect sizes are reported next to p-values because a statistically significant
change can still be too small to matter. Centroid distance is the mean-shift
baseline; a large MMD with a near-zero centroid distance suggests a change in
spread or mixture rather than a simple mean shift.

Distributional tests (`ks`, `mmd`) and the validity/availability test are kept
separate: a quality change is not automatically a distribution shift.

In [ ]:
effect_rows = []
for label, cycle in EVALUATED_CYCLES:
    result = cycle["results"]
    effect_rows.append({
        "cycle": label,
        "ref_mean_len": result.get("ref_mean_length"),
        "cur_mean_len": result.get("cur_mean_length"),
        "length_shift_pct": result.get("length_shift_pct"),
        "centroid_distance": result.get("centroid_distance"),
        "ref_refusal_rate": result.get("ref_refusal_rate"),
        "cur_refusal_rate": result.get("cur_refusal_rate"),
        "refusal_rate_change": result.get("refusal_rate_change"),
        "unique_ref": result.get("reference_unique_responses"),
        "unique_cur": result.get("current_unique_responses"),
        "paired_mmd_stat": fmt(result.get("paired_cluster_mmd_statistic"), 6),
        "matched_prompt_pairs": result.get("paired_cluster_prompt_pairs"),
        "mmd_unit": result.get("mmd_permutation_unit"),
        "mean_latency_s": result.get("provider_latency_mean_seconds"),
    })

if effect_rows:
    display(pd.DataFrame(effect_rows))
else:
    print("No evaluated cycles were produced; effect-size table is empty.")

## 8. Alerts and persistence

A quality alert can fire from a corrected validity change or an absolute
invalid-share guardrail. A drift alert requires a distribution flag in the
current cycle **and** `min_consecutive_cycles` consecutive stored flags.

The persistence history uses the stored `overall_drift_detected` field (any BH
flag). The current-cycle gate separately requires `distribution_shift_detected`,
so a significant availability change alone does not page as drift.

In [ ]:
from sequential_testing import consecutive_streak

stored_results = monitor.storage.get_all_test_results(limit=20)
stored_alerts = monitor.storage.get_all_alerts(limit=20)
stored_flags = [int(row.get("overall_drift_detected") or 0)
                for row in reversed(stored_results)]
streak = consecutive_streak(stored_flags)

print("Stored evaluated results:", len(stored_results))
print("Configured min_consecutive_cycles:", MIN_CONSECUTIVE)
print("Trailing stored-flag streak:", streak)
print("Stored alerts:", len(stored_alerts))

if stored_results:
    alert_history_rows = []
    for row in reversed(stored_results):
        alert_history_rows.append({
            "cycle_id": row.get("cycle_id"),
            "timestamp": row.get("timestamp"),
            "drift_flag_stored": row.get("overall_drift_detected"),
            "quality_guardrail": row.get("overall_quality_degradation"),
            "length_p_adj": fmt(row.get("ks_adjusted_pvalue")),
            "availability_p_adj": fmt(row.get("proportion_z_adjusted_pvalue")),
            "mmd_p_adj": fmt(row.get("mmd_adjusted_pvalue")),
        })
    display(pd.DataFrame(alert_history_rows))

if stored_alerts:
    display(pd.DataFrame([{
        "cycle_id": alert.get("cycle_id"),
        "type": alert.get("alert_type"),
        "severity": alert.get("severity"),
        "message": (alert.get("message") or "")[:220],
    } for alert in reversed(stored_alerts)]))
else:
    print("No alerts fired. With matched stable-model windows this is expected; "
          "it is not a measurement of false-positive rate.")

## 9. Compact history and plots

The cycle summary joins probe rows to detector results by `cycle_id`, so a
reference-only cycle stays visible as baseline collection rather than being
mistaken for a pending evaluation.

In [ ]:
summaries = monitor.storage.get_cycle_summaries(
    limit=20, expected_probes=BATCH_SIZE, reference_window=REF_WINDOW
)
history_rows = []
for summary in reversed(summaries):
    result = summary.get("result") or {}
    history_rows.append({
        "cycle": summary.get("cycle_id"),
        "type": ("reference" if summary.get("reference_baseline") else "evaluation"),
        "probes": summary.get("probe_count"),
        "success": summary.get("success_count"),
        "empty": summary.get("empty_count"),
        "failed": summary.get("failed_count"),
        "tests_run": bool(result),
        "drift_flag": result.get("overall_drift_detected"),
        "length_p_adj": fmt(result.get("ks_adjusted_pvalue")),
        "mmd_p_adj": fmt(result.get("mmd_adjusted_pvalue")),
    })

if history_rows:
    display(pd.DataFrame(history_rows))
else:
    print("No cycles were stored; history table is empty.")

In [ ]:
if detector_rows:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots

    pio.renderers.default = "colab" if "google.colab" in sys.modules else "notebook_connected"
    labels = [row["cycle"] for row in detector_rows]

    pfig = go.Figure()
    for column, name in (
        ("length_p_adj", "length (paired)"),
        ("availability_p_adj", "availability (paired permutation)"),
        ("mmd_p_adj", "MMD (prompt cluster)"),
    ):
        pfig.add_trace(go.Scatter(
            x=labels,
            y=[float(row[column]) if row[column] != "n/a" else None
               for row in detector_rows],
            mode="lines+markers", name=name,
        ))
    pfig.add_hline(y=ALPHA, line_dash="dash", line_color="gray",
                   annotation_text=f"alpha={ALPHA}")
    pfig.update_layout(
        title="BH-adjusted p-values by evaluated cycle (lower = stronger evidence)",
        xaxis_title="cycle", yaxis_title="adjusted p-value",
        yaxis={"range": [0, 1]},
    )
    display(pfig)

    efig = make_subplots(specs=[[{"secondary_y": True}]])
    efig.add_trace(go.Bar(
        x=labels,
        y=[row.get("length_shift_pct") for row in
           [cycle["results"] for _, cycle in EVALUATED_CYCLES]],
        name="length shift (%)",
    ), secondary_y=False)
    efig.add_trace(go.Scatter(
        x=labels,
        y=[row.get("centroid_distance") for row in
           [cycle["results"] for _, cycle in EVALUATED_CYCLES]],
        mode="lines+markers", name="centroid distance",
    ), secondary_y=True)
    efig.update_layout(title="Effect sizes by evaluated cycle")
    efig.update_yaxes(title_text="length shift (%)", secondary_y=False)
    efig.update_yaxes(title_text="centroid distance", secondary_y=True)
    display(efig)

    # Static PNG copies of the same two views. GitHub renders these directly;
    # the interactive Plotly output above stays useful inside Colab.
    import matplotlib.pyplot as plt

    mfig1, ax1 = plt.subplots(figsize=(9, 4))
    for column, name in (
        ("length_p_adj", "length (paired)"),
        ("availability_p_adj", "availability (paired permutation)"),
        ("mmd_p_adj", "MMD (prompt cluster)"),
    ):
        ax1.plot(
            labels,
            [float(row[column]) if row[column] != "n/a" else None
             for row in detector_rows],
            marker="o", label=name,
        )
    ax1.axhline(ALPHA, color="gray", linestyle="--", label=f"alpha={ALPHA}")
    ax1.set_ylim(0, 1)
    ax1.set_xlabel("cycle")
    ax1.set_ylabel("adjusted p-value")
    ax1.set_title("BH-adjusted p-values by evaluated cycle")
    ax1.legend(fontsize=8)
    mfig1.tight_layout()
    display(mfig1)
    plt.close(mfig1)

    mfig2, ax_len = plt.subplots(figsize=(9, 4))
    result_rows = [cycle["results"] for _, cycle in EVALUATED_CYCLES]
    ax_len.bar(labels, [r.get("length_shift_pct") for r in result_rows],
               color="tab:blue", alpha=0.7, label="length shift (%)")
    ax_len.set_ylabel("length shift (%)", color="tab:blue")
    ax_cent = ax_len.twinx()
    ax_cent.plot(labels, [r.get("centroid_distance") for r in result_rows],
                 color="tab:orange", marker="o", label="centroid distance")
    ax_cent.set_ylabel("centroid distance", color="tab:orange")
    ax_len.set_xlabel("cycle")
    ax_len.set_title("Effect sizes by evaluated cycle")
    mfig2.tight_layout()
    display(mfig2)
    plt.close(mfig2)
else:
    print("No evaluated cycles were produced; no detector plots are available.")

## 10. Conclusions and limits

The final cell prints an honest summary of what this run actually produced.
It distinguishes the local Colab experiment from the hosted live Gemini demo
and from controlled simulations.

In [ ]:
print("=== Run summary ===")
print("Repository commit:", commit)
print("Run mode:", run_mode)
print("Simulation:", IS_SIMULATION)
print("Storage:", monitor.storage.info()["backend"],
      "durable=" + str(monitor.storage.info()["durable"]))
print("Cycles attempted:", len(CYCLES))
print("Cycles failed:", len(FAILED_CYCLES))
print("Evaluated cycles:", len(EVALUATED_CYCLES))
print("Stored alerts:", len(monitor.storage.get_all_alerts()))
print("Provider preflight:", "ok" if PROVIDER_OK else "failed/unavailable")

if IS_SIMULATION:
    print("Interpretation: controlled simulation from FakeLLMAdapter. "
          "No live provider behaviour was measured.")
elif not PROVIDER_OK:
    print("Interpretation: no live cycles completed, so no detector result "
          "can be claimed for this run.")
else:
    print("Interpretation: live provider responses from this Colab session, "
          "analysed with the project's own detectors.")

print("Limits: 10 probes and a 30-run reference window are enough to exercise "
      "the workflow, not to estimate detection delay, false-positive rates, "
      "or production alert performance.")
print("Hosted contrast: the deployed demo uses config/config.gemini.yaml with "
      "40 probes, a 120-run reference and PostgreSQL; this notebook does not "
      "use that database or inherit those sample sizes.")